# Give Me Some Credit — Data Validation & Cleaning (Pre-EDA)

This notebook contains **questions, descriptions, and reference answers**.

All tasks use **standardized column names**.


## Task 1: Data Ingestion

**Question:**  
Load the dataset, preserve a raw copy, and report the dataset shape.

**Description:**  
Ensures correct ingestion and reproducibility.


In [23]:
import pandas as pd

DATA_PATH = "../data/cs-training.csv"

df = pd.read_csv(DATA_PATH)
raw_df = df.copy()

df = df.rename(columns={
    'Unnamed: 0': 'unnamed_col',
    'SeriousDlqin2yrs': 'default',
    'RevolvingUtilizationOfUnsecuredLines': 'line_utilization',
    'NumberOfTime30-59DaysPastDueNotWorse': 'dpd_30_60',
    'DebtRatio': 'debt_ratio',
    'MonthlyIncome': 'monthly_income',
    'NumberOfOpenCreditLinesAndLoans': 'n_open_loan_and_line',
    'NumberOfTimes90DaysLate': 'dpd_90',
    'NumberRealEstateLoansOrLines': 'n_real_estate',
    'NumberOfTime60-89DaysPastDueNotWorse': 'dpd_60_90',
    'NumberOfDependents': 'n_dependents'
})

df.shape

(150000, 12)

## Task 2: Data Type Inspection

**Question:**  
Inspect data types and identify schema issues.

**Description:**  
Dtypes determine preprocessing strategy.


In [4]:
df.dtypes

unnamed_col               int64
default                   int64
line_utilization        float64
age                       int64
dpd_30_60                 int64
debt_ratio              float64
monthly_income          float64
n_open_loan_and_line      int64
dpd_90                    int64
n_real_estate             int64
dpd_60_90                 int64
n_dependents            float64
dtype: object

In [5]:
df.head()

,unnamed_col,default,line_utilization,age,dpd_30_60,debt_ratio,monthly_income,n_open_loan_and_line,dpd_90,n_real_estate,dpd_60_90,n_dependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


## Task 3: Target Validation

**Question:**  
Verify the target is binary and non-missing.

**Description:**  
Ensures supervised learning is well-defined.


In [6]:
df["default"].isna().sum()

np.int64(0)

In [7]:
df["default"].value_counts()

default
0    139974
1     10026
Name: count, dtype: int64

## Task 4: Missing Value Detection

**Question:**  
Identify missing values and percentages.

**Description:**  
Quantifies missingness before imputation.


In [24]:
missing = df.isna().sum()
missing = missing[missing > 0]
pct = (missing / len(df) * 100).round(2)
missing.to_frame("count").assign(pct=pct)

,count,pct
monthly_income,29731,19.82
n_dependents,3924,2.62


## Task 5: Logical Validity Checks

**Question:**  
Check for invalid or impossible values.

**Description:**  
Detects domain violations early.


In [6]:
df[df.age < 1].shape[0]

1

In [8]:
(df["age"] <= 0)

0         False
1         False
2         False
3         False
4         False
          ...  
149995    False
149996    False
149997    False
149998    False
149999    False
Name: age, Length: 150000, dtype: bool

In [10]:
for col in ['age', 'monthly_income']:
    result = df[df[col] < 1].shape[0]
    print(col, result)

age 1
monthly_income 1634


In [ ]:
{
    "age <= 0": int((df["age"] <= 0).sum()),
    "monthly_income < 0": int((df["monthly_income"] < 0).sum()),
    "debt_ratio < 0": int((df["debt_ratio"] < 0).sum()),
    "n_dependents < 0": int((df["n_dependents"] < 0).sum()),
}

{'age <= 0': 1,
 'monthly_income < 0': 0,
 'debt_ratio < 0': 0,
 'n_dependents < 0': 0}

## Task 6: Duplicate Check

**Question:**  
Check for duplicate rows.

**Description:**  
Documents data integrity.


In [10]:
df.duplicated().sum()

np.int64(0)

## Task 7: Cardinality Check

**Question:**  
Identify ID-like or near-constant columns.

**Description:**  
High-cardinality features should be excluded.


In [11]:
df.nunique().sort_values(ascending=False)

unnamed_col             150000
line_utilization        125728
debt_ratio              114194
monthly_income           13594
age                         86
n_open_loan_and_line        58
n_real_estate               28
dpd_90                      19
dpd_30_60                   16
dpd_60_90                   13
n_dependents                13
default                      2
dtype: int64

## Task 8: Numerical Sanity Checks

**Question:**  
Check numeric ranges via summary stats.

**Description:**  
Flags extreme values.


In [12]:
df.describe().T[["min", "max", "mean"]]

,min,max,mean
unnamed_col,1.0,150000.0,75000.500000
default,0.0,1.0,0.066840
line_utilization,0.0,50708.0,6.048438
age,0.0,109.0,52.295207
dpd_30_60,0.0,98.0,0.421033
debt_ratio,0.0,329664.0,353.005076
monthly_income,0.0,3008750.0,6670.221237
n_open_loan_and_line,0.0,58.0,8.452760
dpd_90,0.0,98.0,0.265973
n_real_estate,0.0,54.0,1.018240


## Task 9: Feature Grouping

**Question:**  
Group features for later ML preprocessing.

**Description:**  
Prepares for ColumnTransformer.


In [13]:
{
    "target": "default",
    "drop": ["unnamed_col"],
    "continuous": ["line_utilization", "debt_ratio", "monthly_income"],
    "counts": ["age", "dpd_30_60", "dpd_60_90", "dpd_90",
               "n_dependents", "n_open_loan_and_line", "n_real_estate"]
}

{'target': 'default',
 'drop': ['unnamed_col'],
 'continuous': ['line_utilization', 'debt_ratio', 'monthly_income'],
 'counts': ['age',
  'dpd_30_60',
  'dpd_60_90',
  'dpd_90',
  'n_dependents',
  'n_open_loan_and_line',
  'n_real_estate']}

## Task 10: Data Quality Report

**Question:**  
Summarize findings and decisions.

**Description:**  
Final pre-EDA deliverable.


In [14]:
temp = pd.DataFrame({'col1' : [1, 2, 3],
              'col2' : [4, 0, 6]})

temp

,col1,col2
0,1,4
1,2,0
2,3,6


In [16]:
temp.set_index('col1', inplace = True)

In [17]:
temp

,col2
col1,
1,4
2,0
3,6


In [22]:
df.isna().sum().reset_index(name = 'misssing_count')

,index,misssing_count
0,unnamed_col,0
1,default,0
2,line_utilization,0
3,age,0
4,dpd_30_60,0
5,debt_ratio,0
6,monthly_income,29731
7,n_open_loan_and_line,0
8,dpd_90,0
9,n_real_estate,0


In [ ]:
df.query("missing_count > 0")
df[df.missing_count > 0]

In [25]:
df_summary = pd.DataFrame({
    "metric": ["n_rows", "n_cols", "n_duplicates", "n_invalid_age"],
    "value": [
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        (df["age"] <= 0).sum()
    ]
})

df_missing = (
    df.isna()
      .sum()
      .reset_index(name="missing_count")
      .query("missing_count > 0")
      .assign(
          missing_pct=lambda x: x["missing_count"] / len(df) * 100
      )
)

df_decisions = pd.DataFrame({
    "decision_id": range(1, 4),
    "decision": [
        "Drop unnamed_col",
        "Impute monthly_income and n_dependents later",
        "Handle invalid age rows explicitly"
    ]
})

In [28]:
output_path = "..\output\data_quality_report.xlsx"

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    df_summary.to_excel(writer, sheet_name="dataset_summary", index=False)
    df_missing.to_excel(writer, sheet_name="missing_values", index=False)
    df_decisions.to_excel(writer, sheet_name="decisions", index=False)

output_path

<>:1: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
<>:1: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
C:\Users\elihu\AppData\Local\Temp\ipykernel_26476\1229310184.py:1: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
  output_path = "..\output\data_quality_report.xlsx"


'..\\output\\data_quality_report.xlsx'

In [27]:
!pip install xlsxwriter